<a href="https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Rule: Refresh pages with meaningful search visibility that have gone stale

Prioritize content pages that have not been updated recently and still receive meaningful search impressions. Use a transparent score based on staleness and search visibility. The score is for review prioritization, not proof that a refresh will improve performance.

**Signals to check**
- **Staleness:** `days_since_last_update` — linked to FlyRank’s refresh flags.
- **Search visibility:** `impressions_90d` — linked to the volume signal behind quick-win rules.

**Reason codes**
- `stale_and_visible`: the page is at least 180 days since its last update and has at least 500 impressions.
- `visible_but_fresh`: the page has at least 500 impressions but was updated less than 180 days ago.
- `low_visibility_review`: the page does not meet the visibility threshold and needs lower-priority review.

**Action labels:** `refresh_review` or `monitor`.

The signal checks below are exploratory. The rule will use only available page signals, not the decline label or future outcomes.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [14]:

import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path("data/raw/content_refresh_anonymized.csv")

if data_path.exists():
    df = pd.read_csv(data_path)
else:
    df = pd.read_csv(
        "https://raw.githubusercontent.com/"
        "sahanabytes/machine-learning-portfolio/main/"
        "data/raw/content_refresh_anonymized.csv"
    )

required = [
    "content_id", "client_id", "days_since_last_update",
    "impressions_90d", "trend_direction"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}")

df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"], errors="coerce"
)
df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"], errors="coerce"
)
df["decline_audit_only"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 89, 179, 364, np.inf],
    labels=["under_90d", "90_179d", "180_364d", "365d_plus"]
)

df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 99, 499, 2999, np.inf],
    labels=["under_100", "100_499", "500_2999", "3000_plus"]
)

print("SIGNAL 1: STALENESS — n and observed decline rate")
staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          decline_rate=("decline_audit_only", "mean")
      )
      .reset_index()
)
print(staleness_table.to_string(index=False))

print("\nSIGNAL 2: SEARCH VISIBILITY — n and observed decline rate")
visibility_table = (
    df.groupby("visibility_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          decline_rate=("decline_audit_only", "mean")
      )
      .reset_index()
)
print(visibility_table.to_string(index=False))


SIGNAL 1: STALENESS — n and observed decline rate
staleness_bucket     n  decline_rate
       under_90d 20655      0.512031
         90_179d  9171      0.611057
        180_364d   169      0.467456
       365d_plus     5      0.600000

SIGNAL 2: SEARCH VISIBILITY — n and observed decline rate
visibility_bucket    n  decline_rate
        under_100 7994      0.389042
          100_499 5280      0.604356
         500_2999 8443      0.620632
        3000_plus 8283      0.569963


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [16]:

from pathlib import Path
import numpy as np
import pandas as pd

queue = df.copy()

queue["days_since_last_update"] = pd.to_numeric(
    queue["days_since_last_update"], errors="coerce"
)
queue["impressions_90d"] = pd.to_numeric(
    queue["impressions_90d"], errors="coerce"
)

# Rule: prioritize stale pages with meaningful search visibility.
queue["baseline_score"] = (
    (queue["days_since_last_update"].fillna(0) >= 180).astype(int) * 50
    + (queue["impressions_90d"].fillna(0) >= 500).astype(int) * 50
)

queue["reason_code"] = np.select(
    [
        (queue["days_since_last_update"] >= 180)
        & (queue["impressions_90d"] >= 500),
        (queue["days_since_last_update"] < 180)
        & (queue["impressions_90d"] >= 500),
    ],
    ["stale_and_visible", "visible_but_fresh"],
    default="low_visibility_review"
)

queue["action"] = np.where(
    queue["reason_code"] == "stale_and_visible",
    "refresh_review",
    "monitor"
)

queue = queue.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue.insert(0, "rank", range(1, len(queue) + 1))

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

print("Queue rows:", len(queue))
print("CSV written to:", output_path)
print("\nAction counts:")
print(queue["action"].value_counts().to_string())
print("\nTop 10 preview:")
print(queue[[
    "rank", "content_id", "baseline_score",
    "reason_code", "action", "days_since_last_update",
    "impressions_90d"
]].head(10).to_string(index=False))


Queue rows: 30000
CSV written to: work/outputs/baseline_action_score.csv

Action counts:
action
monitor           29983
refresh_review       17

Top 10 preview:
 rank           content_id  baseline_score       reason_code         action  days_since_last_update  impressions_90d
    1 content_cf56e2e2e282             100 stale_and_visible refresh_review                     194            61678
    2 content_7368877ea310             100 stale_and_visible refresh_review                     194            59472
    3 content_1bfaa38ff26c             100 stale_and_visible refresh_review                     194            25715
    4 content_0a91db491d14             100 stale_and_visible refresh_review                     193            13299
    5 content_5feee3994adb             100 stale_and_visible refresh_review                     194             7812
    6 content_c2d929d83eaa             100 stale_and_visible refresh_review                     193             7558
    7 content_b16bd7

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [18]:

top20 = queue.head(20).copy()

def review_row(row):
    if row["reason_code"] == "stale_and_visible":
        confidence = "Moderate: stale and visible; refresh benefit unproven."
        could_be_wrong = (
            "The page may already perform well, or its update date "
            "may not reflect meaningful content changes."
        )
    elif row["reason_code"] == "visible_but_fresh":
        confidence = "Low: visibility is present, but the page is relatively fresh."
        could_be_wrong = (
            "The page may need no refresh; impressions alone do not "
            "show a content problem."
        )
    else:
        confidence = "Low: limited evidence for prioritizing a refresh."
        could_be_wrong = (
            "The page may have value not captured by impressions or "
            "the available update-age field."
        )
    return pd.Series([confidence, could_be_wrong])

top20[["confidence_note", "what_would_make_it_wrong"]] = top20.apply(
    review_row, axis=1
)

print(top20[[
    "rank", "content_id", "action", "reason_code",
    "confidence_note", "what_would_make_it_wrong"
]].to_string(index=False))


 rank           content_id         action       reason_code                                               confidence_note                                                                          what_would_make_it_wrong
    1 content_cf56e2e2e282 refresh_review stale_and_visible        Moderate: stale and visible; refresh benefit unproven. The page may already perform well, or its update date may not reflect meaningful content changes.
    2 content_7368877ea310 refresh_review stale_and_visible        Moderate: stale and visible; refresh benefit unproven. The page may already perform well, or its update date may not reflect meaningful content changes.
    3 content_1bfaa38ff26c refresh_review stale_and_visible        Moderate: stale and visible; refresh benefit unproven. The page may already perform well, or its update date may not reflect meaningful content changes.
    4 content_0a91db491d14 refresh_review stale_and_visible        Moderate: stale and visible; refresh benefit unproven

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [20]:
# Section 4 — Weak picks + leakage check

print("WEAK PICKS — bottom 10 rows in the ranked queue")
print(
    queue.tail(10)[[
        "rank", "content_id", "baseline_score",
        "reason_code", "action",
        "days_since_last_update", "impressions_90d"
    ]].to_string(index=False)
)

print("\nLEAKAGE CHECK")
score_inputs = ["days_since_last_update", "impressions_90d"]
excluded_audit_fields = [
    col for col in ["trend_direction", "trend_pct", "decline_audit_only"]
    if col in queue.columns
]

print("Score inputs:", score_inputs)
print("Trend/decline audit fields excluded from scoring:", excluded_audit_fields)
print("Product flags used in scoring: None")
print("Future outcome windows used in scoring: None")

assert all(col in queue.columns for col in score_inputs)
assert "trend_direction" not in score_inputs
assert "trend_pct" not in score_inputs
assert "decline_audit_only" not in score_inputs

print("\nLeakage check passed: the score uses only the two stated inputs.")

WEAK PICKS — bottom 10 rows in the ranked queue
 rank           content_id  baseline_score           reason_code  action  days_since_last_update  impressions_90d
29991 content_a97955b77e18               0 low_visibility_review monitor                       8                1
29992 content_d557b826e19f               0 low_visibility_review monitor                      20                1
29993 content_4f11749a1306               0 low_visibility_review monitor                      20                1
29994 content_cc19609c216e               0 low_visibility_review monitor                      20                1
29995 content_321383d0e9cf               0 low_visibility_review monitor                      20                1
29996 content_6bca373d8751               0 low_visibility_review monitor                      20                1
29997 content_7d12e9e7a4a5               0 low_visibility_review monitor                      20                1
29998 content_901b40631379              

### Weak picks and leakage check

- **Weak picks:** Pages ranked near the bottom receive lower scores and are assigned `monitor`. This is a prioritization rule, not proof that these pages have no improvement opportunity.
- **Limitation:** The score uses only update age and 90-day impressions. It does not establish that refreshing a page will improve performance.
- **Leakage check:** The scoring rule uses `days_since_last_update` and `impressions_90d` only. Trend and decline fields were used for signal inspection, not as scoring inputs. No product flags or future outcome windows are used in the score.
- **Next step:** Review the recommended pages manually before deciding to refresh them.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.